### AI Financial Analyst

#### Member 2

### Notebook 4 - Text Preprocessing

---

#### Objective

The objective of this notebook is to prepare the extracted SEC narrative
sections for embedding generation.

The preprocessing pipeline includes:

- Cleaning narrative text
- Removing unnecessary formatting
- Normalising whitespace
- Splitting long narratives into manageable text chunks

The processed chunks generated in this notebook will be used as input for
embedding generation and the FAISS vector database.

---

#### CRISP-DM Phase

**Data Preparation**

#### Step 1 - Importing Libraries

In [1]:
# Importing Libraries

import re
import pandas as pd

print("Libraries imported successfully.")

Libraries imported successfully.


#### Step 2 - Loading Narrative Dataset

In [2]:
# Loading Narrative Dataset

narrative_df = pd.read_parquet(
    "../data/processed/narrative_sections.parquet"
)

print("Narrative dataset loaded successfully.")

display(narrative_df.head())

Narrative dataset loaded successfully.


,ticker,company_name,cik,year,section_1,section_1A,section_7
0,APTV,Aptiv PLC,1521332,2014,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
1,APTV,Aptiv PLC,1521332,2015,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
2,APTV,Aptiv PLC,1521332,2016,"ITEM 1. BUSINESS\n“Delphi,” the “Company,” “we...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
3,APTV,Aptiv PLC,1521332,2018,"ITEM 1. BUSINESS\n“Aptiv,” the “Company,” “we,...",ITEM 1A. RISK FACTORS\nSet forth below are cer...,ITEM 7. MANAGEMENT’S DISCUSSION AND ANALYSIS O...
4,ARTNA,ARTESIAN RESOURCES CORP,863110,2014,ITEM 1. BUSINESS\nGeneral Information\nArtesia...,ITEM 1A. RISK FACTORS\nWe are exposed to a var...,ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS O...


#### Step 3 - Validating Narrative Dataset

Before preprocessing begins, the extracted narrative dataset is validated
to ensure that all required variables are available.

In [3]:
# Validating Dataset

print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

print("Rows :", len(narrative_df))
print("Columns :", len(narrative_df.columns))

display(narrative_df.columns)

DATASET SUMMARY
Rows : 109
Columns : 7


Index(['ticker', 'company_name', 'cik', 'year', 'section_1', 'section_1A',
       'section_7'],
      dtype='object')

#### Step 4 - Previewing Narrative Text


In [4]:
# Previewing Narrative Text

sample = narrative_df.iloc[0]

print("=" * 60)
print("SECTION 1A PREVIEW")
print("=" * 60)

print(sample["section_1A"][:1000])

SECTION 1A PREVIEW
ITEM 1A. RISK FACTORS
Set forth below are certain risks and uncertainties that could adversely affect our results of operations or financial condition and cause our actual results to differ materially from those expressed in forward-looking statements made by the Company. Also refer to the Cautionary Statement Regarding Forward-Looking Information in this annual report.
Risks Related to Business Environment and Economic Conditions
The cyclical nature of automotive sales and production can adversely affect our business.
Our business is directly related to automotive sales and automotive vehicle production by our customers. Automotive sales and production are highly cyclical and, in addition to general economic conditions, also depend on other factors, such as consumer confidence and consumer preferences. Lower global automotive sales would be expected to result in substantially all of our automotive OEM customers lowering vehicle production schedules, which has a dire

#### Step 5 - Cleaning Narrative Text

The extracted narrative text may contain unnecessary line breaks,
multiple spaces and formatting artefacts inherited from the original
SEC filings.

These elements are removed to create a consistent text format suitable
for chunking and embedding generation.

In [5]:
# Text Cleaning Function

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Remove leading/trailing spaces
    text = text.strip()

    return text

#### Step 6 - Cleaning Narrative Sections

In [6]:
# Cleaning Narrative Sections

NARRATIVE_SECTIONS = [
    "section_1",
    "section_1A",
    "section_7"
]

clean_df = narrative_df.copy()

for section in NARRATIVE_SECTIONS:
    clean_df[section] = clean_df[section].apply(clean_text)

print("Narrative text cleaned successfully.")

Narrative text cleaned successfully.


#### Step 7 -  Validating Cleaned Text


In [7]:
#  Validating Cleaned Text

print("=" * 60)
print("CLEANED TEXT PREVIEW")
print("=" * 60)

print(clean_df.loc[0, "section_1A"][:1000])

CLEANED TEXT PREVIEW
ITEM 1A. RISK FACTORS Set forth below are certain risks and uncertainties that could adversely affect our results of operations or financial condition and cause our actual results to differ materially from those expressed in forward-looking statements made by the Company. Also refer to the Cautionary Statement Regarding Forward-Looking Information in this annual report. Risks Related to Business Environment and Economic Conditions The cyclical nature of automotive sales and production can adversely affect our business. Our business is directly related to automotive sales and automotive vehicle production by our customers. Automotive sales and production are highly cyclical and, in addition to general economic conditions, also depend on other factors, such as consumer confidence and consumer preferences. Lower global automotive sales would be expected to result in substantially all of our automotive OEM customers lowering vehicle production schedules, which has a di

#### Step 8 - Chunk Narrative Text

The cleaned narrative sections are divided into smaller overlapping chunks.

Chunking improves retrieval performance by allowing the embedding model to
represent smaller, semantically coherent pieces of text.

For this project:

- Chunk size = 300 words
- Overlap = 50 words

The overlap helps preserve context between consecutive chunks.

In [8]:
# Chunking Function

def create_chunks(text, chunk_size=300, overlap=50):
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

#### Step 9 - Generating Narrative Chunks

In [9]:
# Generating Narrative Chunks

chunk_records = []

for _, row in clean_df.iterrows():
    for section in ["section_1", "section_1A", "section_7"]:
        chunks = create_chunks(row[section])
        for chunk_id, chunk in enumerate(chunks):
            chunk_records.append({
                "ticker": row["ticker"],
                "company_name": row["company_name"],
                "cik": row["cik"],
                "year": row["year"],
                "section": section,
                "chunk_id": chunk_id,
                "chunk_text": chunk
            })

chunks_df = pd.DataFrame(chunk_records)

print("Chunk generation completed.")

Chunk generation completed.


#### Step 10  - Chunk Dataset Summary


In [10]:
# Chunk Dataset Summary

print("=" * 60)
print("CHUNK DATASET SUMMARY")
print("=" * 60)

print("Total Chunks :", len(chunks_df))
print("Companies :", chunks_df["ticker"].nunique())

display(chunks_df.head())

CHUNK DATASET SUMMARY
Total Chunks : 10009
Companies : 22


,ticker,company_name,cik,year,section,chunk_id,chunk_text
0,APTV,Aptiv PLC,1521332,2014,section_1,0,"ITEM 1. BUSINESS “Delphi,” the “Company,” “we,..."
1,APTV,Aptiv PLC,1521332,2014,section_1,1,"focus on these markets, particularly China, wh..."
2,APTV,Aptiv PLC,1521332,2014,section_1,2,"PBGC were redeemed, respectively, for approxim..."
3,APTV,Aptiv PLC,1521332,2014,section_1,3,engine management systems including fuel handl...
4,APTV,Aptiv PLC,1521332,2014,section_1,4,and products. Our customer base includes all 2...


In [11]:
print("Years :", sorted(chunks_df["year"].unique()))
print("Sections :", chunks_df["section"].unique())

Years : ['2014', '2015', '2016', '2017', '2018']
Sections : ['section_1' 'section_1A' 'section_7']


#### Step 10A - Chunk Distribution

The number of generated chunks is summarised for each narrative section.

This provides an indication of how the chunking process has distributed the
narrative content across the three SEC filing sections.

In [12]:
print("=" * 60)
print("CHUNK DISTRIBUTION")
print("=" * 60)

distribution = (
    chunks_df
    .groupby("section")
    .size()
    .reset_index(name="Number of Chunks")
)

display(distribution)

CHUNK DISTRIBUTION


,section,Number of Chunks
0,section_1,2250
1,section_1A,3693
2,section_7,4066


#### Step 11 - Chunk Statistics

In [13]:
# Chunk Statistics

chunks_df["word_count"] = (
    chunks_df["chunk_text"]
    .str.split()
    .str.len()
)

print("=" * 60)
print("CHUNK STATISTICS")
print("=" * 60)

print("Average Words :", round(chunks_df["word_count"].mean()))
print("Minimum Words :", chunks_df["word_count"].min())
print("Maximum Words :", chunks_df["word_count"].max())

CHUNK STATISTICS
Average Words : 294
Minimum Words : 1
Maximum Words : 300


In [14]:
duplicates = chunks_df.duplicated().sum()
print("Duplicate Chunks :", duplicates)

Duplicate Chunks : 0


#### Step 12 - Saving Chunk Dataset

In [15]:
# Saving Chunk Dataset

csv_path = "../data/processed/processed_chunks.csv"
parquet_path = "../data/processed/processed_chunks.parquet"

chunks_df.to_csv(csv_path, index=False)
chunks_df.to_parquet(parquet_path, index=False)

print("=" * 60)
print("FILES SAVED")
print("=" * 60)

print(csv_path)
print(parquet_path)

FILES SAVED
../data/processed/processed_chunks.csv
../data/processed/processed_chunks.parquet


#### Conclusion

This notebook successfully prepared the extracted SEC narrative sections
for embedding generation.

The preprocessing pipeline included:

- Text cleaning
- Whitespace normalisation
- Word-based chunk generation with overlap

The resulting chunk dataset was validated and saved in both CSV and
Parquet formats.

These processed narrative chunks will be used in Notebook 5 to generate
sentence embeddings and construct the FAISS vector database that supports
the Retrieval-Augmented Generation (RAG) pipeline.